# Multimodal speech emotion recognition on MELD: project results

**Task.** Classify the emotion of a spoken utterance from the TV series *Friends* (MELD dataset, 7 classes:
neutral, joy, surprise, anger, sadness, disgust, fear) from the **audio** and from an **automatic transcript**
of that audio (Whisper), and test whether fusing the two beats either modality alone.

**How to read this notebook.** Every number comes from the run directories under `runs/` written by
`bash scripts/run_all.sh`: each model is trained on the train split, tuned only on dev, and evaluated on the
test split **exactly once**. Scores are means over the training seeds; 95 % confidence intervals come from
bootstrapping the test utterances. The main metric is **weighted F1** (the MELD standard); macro F1 and the
recall of the rare classes fear and disgust show how the model treats minority emotions.

Contents: 1 pipeline, 2 data, 3 ASR quality, 4 main results, 5 training curves, 6 per-class results,
7 significance, 8 robustness, 9 ablations and overfitting, 10 explanations, 11 quality gates, 12 reproduce.

In [ ]:
# parameters. To try the notebook on the 48-row smoke run instead of the full run, use
# RUNS_ROOT = 'data/processed/smoke_runs', REPORTS_DIR = 'data/processed/smoke_reports',
# MANIFEST_DIR = 'data/processed/smoke_manifests' (smoke numbers are plumbing checks, not results).
RUNS_ROOT = 'runs'
REPORTS_DIR = 'reports'
MANIFEST_DIR = None   # None = data/manifests
SEED = 13             # seed whose training curves and explanations are shown

In [ ]:
import json
import sys
import textwrap
from pathlib import Path

ROOT =Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyBboxPatch, Patch

from src import evaluate as ev
from src.common import EMOTIONS, SPLITS, load_yaml
from src.data.dataset import load_manifest

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 90)
runs_root = ev.resolve_path(RUNS_ROOT)
reports = ev.resolve_path(REPORTS_DIR)

# Chart style: recessive axes and grid, text in ink colours, one fixed colour per input modality
# (validated colour-blind-safe order). Every chart is followed by its numbers as a table.
INK, INK_2, GRID, SURFACE = '#0b0b0b', '#52514e', '#e4e3df', '#fcfcfb'
SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4']   # fixed order, never cycled
MODALITY_COLOR = {'text': SERIES[0], 'audio': SERIES[1], 'fusion': SERIES[2], 'none': '#a3a29c'}
MODALITY_NAME = {'none': 'baseline (no learned modality)', 'text': 'text (transcript)', 'audio': 'audio',
                 'fusion': 'audio + text fusion'}
BLUES = LinearSegmentedColormap.from_list(
    'blues', ['#f4f8fe', '#cde2fb', '#86b6ef', '#3987e5', '#256abf', '#184f95', '#0d366b'])
plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE, 'figure.dpi': 110,
    'axes.edgecolor': GRID, 'axes.labelcolor': INK_2, 'xtick.color': INK_2, 'ytick.color': INK_2, 'text.color': INK,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.color': GRID,
    'grid.linewidth': 0.8, 'axes.axisbelow': True, 'font.size': 10, 'axes.titlesize': 11,
    'axes.titleweight': 'bold', 'axes.titlecolor': INK, 'legend.frameon': False, 'lines.linewidth': 2})

LABELS = {
    'majority': '0 Majority class', 'shortcut_lr': '1 Duration + length (LR)',
    'text_tfidf': '2 TF-IDF text (LR)', 'text_tfidf_gold': '2 TF-IDF text (LR), gold text',
    'audio_egemaps_lr': '3 eGeMAPS audio (LR)', 'audio_egemaps_mlp': '3 eGeMAPS audio (MLP)',
    'text_roberta': '4 RoBERTa text', 'text_roberta_gold': '4 RoBERTa text, gold text',
    'audio_wavlm_cached': '5 WavLM audio, frozen', 'audio_wavlm_ft': '5 WavLM audio, fine-tuned',
    'fusion_late': '6 Late fusion', 'fusion_gated': '7 Gated fusion', 'fusion_xattn': '8 Cross-attention fusion',
    'fusion_gated_gold': '7 Gated fusion, gold text',
}


def label(name):
    return LABELS.get(name, name)


test_df = ev.load_run_results(runs_root)   # one row per test-evaluated run
dev_df = ev.load_dev_results(runs_root)    # one row per trained run (dev scores)
assert not test_df.empty, f'no test_metrics.json under {runs_root}: run scripts/run_all.sh (stage 7) first'
cw = ev.class_weight_table(dev_df, test_df)
FUSION, fusion_candidates, fusion_reason = ev.select_fusion(dev_df, test_df, 'auto', ev.rejected_families(cw))
UNIMODAL = ['text_roberta', 'audio_wavlm_ft']
print(f'{len(test_df)} test-evaluated runs of {test_df["name"].nunique()} models, seeds '
      f'{sorted(test_df["seed"].astype(str).unique())}; {len(dev_df)} trained runs in total')
print(f'final fusion model (chosen on dev): {label(FUSION)}')

## 1 Pipeline

The audio goes two ways: Whisper turns it into a transcript for the text encoder, and WavLM encodes the
waveform directly. The fusion models combine the two embeddings. Models 0-3 are simple baselines that show
how far cheap shortcuts get.

In [ ]:
def box(ax, x, y, w, h, text, fill):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.02,rounding_size=0.12', fc=fill, ec='none'))
    ax.text(x + w / 2, y + h / 2, text, ha='center', va='center', fontsize=9.5, color=INK)


def arrow(ax, start, end):
    ax.annotate('', xy=end, xytext=start, arrowprops=dict(arrowstyle='-|>', color=INK_2, lw=1.4))


fig, ax = plt.subplots(figsize=(12, 3.8))
ax.set_xlim(0, 12); ax.set_ylim(-0.4, 3.6); ax.axis('off')
GRAY_T, BLUE_T, ORANGE_T, AQUA_T = '#efeeeb', '#cde2fb', '#fad9cc', '#c6ebde'
box(ax, 0.1, 1.4, 2.0, 1.0, 'Speech clip\n(MELD, 16 kHz mono)', GRAY_T)
box(ax, 2.7, 2.5, 1.9, 0.9, 'Whisper ASR\n(transcript)', BLUE_T)
box(ax, 5.1, 2.5, 2.0, 0.9, 'RoBERTa\ntext encoder', BLUE_T)
box(ax, 2.7, 0.4, 4.4, 0.9, 'WavLM audio encoder\n(13 layers, learned layer weights)', ORANGE_T)
box(ax, 7.6, 1.4, 2.7, 1.0, 'Fusion\nlate / gated / cross-attention', AQUA_T)
box(ax, 10.6, 1.4, 1.3, 1.0, '7 emotions', GRAY_T)
arrow(ax, (2.1, 2.05), (2.7, 2.9)); arrow(ax, (2.1, 1.75), (2.7, 0.85))
arrow(ax, (4.6, 2.95), (5.1, 2.95)); arrow(ax, (7.1, 2.95), (7.6, 2.15)); arrow(ax, (7.1, 0.85), (7.6, 1.65))
arrow(ax, (10.3, 1.9), (10.6, 1.9))
ax.text(0.1, -0.25, 'Baselines (models 0-3): majority class, clip duration + transcript length, TF-IDF on the transcript, '
        'eGeMAPS acoustic features.', fontsize=9, color=INK_2)
plt.show()

## 2 Data

MELD utterances after the data policy (duplicates and sub-0.5 s train fragments removed). The classes are
heavily imbalanced: neutral dominates, fear and disgust are rare, which is why the report tracks macro F1 and
minority-class recall next to weighted F1.

In [ ]:
counts = pd.DataFrame({s: load_manifest(s, manifest_dir_override=MANIFEST_DIR)['emotion'].value_counts()
                       for s in SPLITS}).reindex(EMOTIONS).fillna(0).astype(int)
share = counts / counts.sum()
fig, ax = plt.subplots(figsize=(10, 3.6))
x, w = np.arange(len(EMOTIONS)), 0.27
for i, s in enumerate(SPLITS):
    ax.bar(x + (i - 1) * w, 100 * share[s], w - 0.03, color=SERIES[i], label=f'{s} (n={counts[s].sum():,})')
ax.set_xticks(x, EMOTIONS); ax.grid(axis='x', visible=False)
ax.set_ylabel('% of utterances in the split'); ax.set_title('Class distribution per split')
ax.legend(ncols=3, loc='upper right')
plt.show()
table = counts.copy()
for s in SPLITS:
    table[f'{s} %'] = (100 * share[s]).round(1)
table.loc['total'] = table.sum().where(~table.columns.str.endswith('%'), 100.0)
table

## 3 ASR quality

The text models read Whisper transcripts, not the gold subtitles, so transcription errors are part of the
task. Word error rate (WER) is computed against the gold MELD text after normalisation.

In [ ]:
from src.asr import quality

asr_tag = test_df['test_metrics'].map(lambda t: t.get('asr_tag')).dropna()
asr_tag = asr_tag.iloc[0] if len(asr_tag) else load_yaml('configs/asr.yaml')['tag']
try:
    asr_rows = quality.load_rows(asr_tag, MANIFEST_DIR)
    display(Markdown(f'ASR model tag: `{asr_tag}`'))
    display(quality.wer_per_split(asr_rows).style.format({'WER': '{:.1%}', 'WER_policy_kept': '{:.1%}'}).hide(axis='index'))
    per_class = quality.wer_per_class(asr_rows)[[s for s in SPLITS if s in asr_rows['split'].unique()]]
    display(per_class.style.format('{:.1%}', na_rep='').set_caption('WER per class and split'))
except FileNotFoundError as e:
    print(f'no ASR transcripts found ({e}); run stage 1 of scripts/run_all.sh')

## 4 Main results (test split)

Models are listed in ladder order: baselines first, then unimodal deep models, then fusion. Bars show the mean
over seeds; whiskers show the 95 % bootstrap confidence interval (mean of the per-seed bounds). The final
fusion model is chosen on **dev** weighted F1, never on test.

In [ ]:
g = test_df.groupby('name', sort=False)
res = g.agg(modality=('modality', 'first'), seeds=('seed', 'nunique'), accuracy=('accuracy', 'mean'),
            weighted_f1=('weighted_f1', 'mean'), weighted_f1_std=('weighted_f1', 'std'),
            wf1_lo=('weighted_f1_lo', 'mean'), wf1_hi=('weighted_f1_hi', 'mean'),
            macro_f1=('macro_f1', 'mean'), mf1_lo=('macro_f1_lo', 'mean'), mf1_hi=('macro_f1_hi', 'mean'),
            fear_recall=('fear_recall', 'mean'), disgust_recall=('disgust_recall', 'mean'),
            dev_weighted_f1=('dev_weighted_f1', 'mean')).reset_index()
res['model'] = res['name'].map(label)

fig, axes = plt.subplots(1, 2, figsize=(12, 0.42 * len(res) + 1.4), sharey=True)
y = np.arange(len(res))
for ax, (col, lo, hi, title) in zip(axes, [('weighted_f1', 'wf1_lo', 'wf1_hi', 'Weighted F1 (main metric)'),
                                           ('macro_f1', 'mf1_lo', 'mf1_hi', 'Macro F1 (all classes equal)')]):
    colors = res['modality'].map(MODALITY_COLOR).fillna(MODALITY_COLOR['none'])
    err = np.vstack([(res[col] - res[lo]).clip(lower=0), (res[hi] - res[col]).clip(lower=0)])
    ax.barh(y, res[col], height=0.68, color=colors, xerr=err, error_kw=dict(ecolor=INK_2, lw=1, capsize=2))
    for yi, v, h in zip(y, res[col], res[hi].fillna(res[col])):
        ax.text(h + 0.01, yi, f'{v:.3f}', va='center', fontsize=8.5, color=INK)
    ax.set_title(title); ax.set_xlim(0, min(1.0, float(res[hi].max(skipna=True) or res[col].max()) + 0.12))
    ax.grid(axis='y', visible=False)
axes[0].set_yticks(y, res['model']); axes[0].invert_yaxis()
present = [m for m in MODALITY_COLOR if m in set(res['modality'])]
fig.legend([Patch(color=MODALITY_COLOR[m]) for m in present], [MODALITY_NAME[m] for m in present],
           ncols=len(present), loc='lower center', bbox_to_anchor=(0.5, -0.02))
fig.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()

shown = res[['model', 'seeds', 'accuracy', 'weighted_f1', 'weighted_f1_std', 'wf1_lo', 'wf1_hi', 'macro_f1',
             'mf1_lo', 'mf1_hi', 'fear_recall', 'disgust_recall', 'dev_weighted_f1']]
shown.columns = ['Model', 'Seeds', 'Accuracy', 'Weighted F1', '± std (seeds)', 'wF1 CI low', 'wF1 CI high', 'Macro F1',
                 'mF1 CI low', 'mF1 CI high', 'Fear recall', 'Disgust recall', 'Dev weighted F1']
shown.style.format(precision=3, na_rep='-').hide(axis='index')

### Key findings (generated from the numbers above)

In [ ]:
mc, best_uni = ev.mcnemar_table(test_df, runs_root, FUSION, UNIMODAL) if FUSION else (pd.DataFrame(), None)
r = res.set_index('name')
lines = []
if 'majority' in r.index:
    lines.append(f'- The majority-class baseline reaches weighted F1 {r.loc["majority", "weighted_f1"]:.3f} '
                 f'(accuracy {r.loc["majority", "accuracy"]:.3f}): the floor every model must beat.')
for u in UNIMODAL:
    if u in r.index:
        lines.append(f'- {label(u)}: weighted F1 {r.loc[u, "weighted_f1"]:.3f}, macro F1 {r.loc[u, "macro_f1"]:.3f}.')
if FUSION in r.index:
    f = r.loc[FUSION]
    lines.append(f'- Final model **{label(FUSION)}** (chosen on dev): weighted F1 {f["weighted_f1"]:.3f} '
                 f'[{f["wf1_lo"]:.3f}, {f["wf1_hi"]:.3f}], macro F1 {f["macro_f1"]:.3f}, accuracy {f["accuracy"]:.3f}, '
                 f'fear recall {f["fear_recall"]:.3f}, disgust recall {f["disgust_recall"]:.3f}.')
    if best_uni in r.index:
        gain = f['weighted_f1'] - r.loc[best_uni, 'weighted_f1']
        lines.append(f'- Against the best unimodal model on dev ({label(best_uni)}), the fusion model changes test '
                     f'weighted F1 by {gain:+.3f}.')
        m = mc[(mc['fusion'] == FUSION) & (mc['unimodal'] == best_uni)] if len(mc) else mc
        if len(m):
            ps = ', '.join(f'seed {s}: p = {p:.3g}' for s, p in zip(m['seed'], m['p_value']))
            sig = 'significant' if (m['p_value'] < 0.05).all() else 'not significant on every seed'
            lines.append(f'- McNemar test on the paired test predictions: {ps} ({sig} at α = 0.05).')
display(Markdown('\n'.join(lines) or '_no results yet_'))

## 5 Training curves

One row per deep model (seed set in the parameters cell). The dashed line marks the epoch whose checkpoint was
kept: training stops early when dev weighted F1 has not improved for 3 epochs.

- **Train loss** is the class-weighted (square-root inverse-frequency) cross-entropy that the optimiser minimises.
- **Validation loss is not logged per epoch**: model selection uses dev weighted F1. The table below gives the dev
  cross-entropy of the kept checkpoint, computed from its saved dev predictions; it is unweighted, so compare it
  with care to the weighted train loss.

In [ ]:
DEEP = [n for n in ['text_roberta', 'audio_wavlm_cached', 'audio_wavlm_ft', 'fusion_gated', 'fusion_xattn']
        if (runs_root / n / str(SEED) / 'metrics.json').exists()]
if not DEEP:
    print(f'no deep-model runs for seed {SEED} under {runs_root}')
else:
    fig, axes = plt.subplots(len(DEEP), 3, figsize=(12, 2.5 * len(DEEP)), squeeze=False)
    rows = []
    for i, name in enumerate(DEEP):
        run = runs_root / name / str(SEED)
        m = json.loads((run / 'metrics.json').read_text())
        h = pd.DataFrame(m['history'])
        best = m['best']['epoch']
        ax_l, ax_a, ax_f = axes[i]
        ax_l.plot(h['epoch'], h['train_loss'], 'o-', color=SERIES[0], ms=6, label='train')
        for ax, col in ((ax_a, 'accuracy'), (ax_f, 'weighted_f1')):
            ax.plot(h['epoch'], h[f'train_{col}'], 'o-', color=SERIES[0], ms=6, label='train')
            ax.plot(h['epoch'], h[f'dev_{col}'], 's-', color=SERIES[1], ms=6, label='dev')
        for ax in axes[i]:
            ax.axvline(best, color=INK_2, ls='--', lw=1)
            ax.set_xticks(h['epoch'])
        ax_l.set_ylabel(textwrap.fill(label(name), 16), rotation=0, ha='right', va='center', color=INK)
        p = pd.read_csv(run / 'predictions.csv')
        probs = p[[f'p_{c}' for c in range(len(EMOTIONS))]].to_numpy()
        dev_ce = float(-np.log(np.clip(probs[np.arange(len(p)), p['label'].to_numpy()], 1e-12, 1)).mean())
        b = h.loc[h['epoch'] == best].iloc[0]
        rows.append({'Model': label(name), 'Epochs run': len(h), 'Best epoch': best, 'Train loss @best': b['train_loss'],
                     'Dev CE @best (unweighted)': dev_ce, 'Train acc @best': b['train_accuracy'],
                     'Dev acc @best': b['dev_accuracy'], 'Train wF1 @best': b['train_weighted_f1'],
                     'Dev wF1 @best': b['dev_weighted_f1'], 'Train - dev wF1': b['train_weighted_f1'] - b['dev_weighted_f1']})
    for ax, t in zip(axes[0], ['Train loss (weighted CE)', 'Accuracy', 'Weighted F1']):
        ax.set_title(t)
    axes[0, 1].legend(loc='best')
    for ax in axes[-1]:
        ax.set_xlabel('epoch')
    fig.tight_layout()
    plt.show()
    display(pd.DataFrame(rows).style.format(precision=3).hide(axis='index'))

## 6 Per-class results

Test F1 per emotion (mean over seeds). Rare classes (fear, disgust) are the hard part of MELD; the confusion
matrix of the final fusion model shows where its errors go (rows = true class, each row sums to 100 %).

In [ ]:
pc = ev.per_class_table(test_df).set_index('model')
pc.index = [label(n) for n in pc.index]
fig, ax = plt.subplots(figsize=(10, 0.42 * len(pc) + 1.3))
im = ax.imshow(pc.to_numpy(dtype=float), cmap=BLUES, vmin=0, vmax=1, aspect='auto')
for (i, j), v in np.ndenumerate(pc.to_numpy(dtype=float)):
    if not np.isnan(v):
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8.5, color='white' if v > 0.55 else INK)
ax.set_xticks(range(len(EMOTIONS)), EMOTIONS); ax.set_yticks(range(len(pc)), pc.index)
ax.grid(False); ax.set_title('Test F1 per class')
fig.colorbar(im, ax=ax, fraction=0.03, label='F1')
plt.show()
support = test_df['test_metrics'].iloc[0]['test'].get('per_class', {})
print('test support per class:', {e: support.get(e, {}).get('support') for e in EMOTIONS})

In [ ]:
if FUSION:
    mats = [np.asarray(tm['test']['confusion_row_norm']) for tm in test_df.loc[test_df['name'] == FUSION, 'test_metrics']]
    cm = np.mean(mats, axis=0)
    fig, ax = plt.subplots(figsize=(6.2, 5.2))
    im = ax.imshow(cm, cmap=BLUES, vmin=0, vmax=1)
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, f'{100 * v:.0f}', ha='center', va='center', fontsize=8.5, color='white' if v > 0.55 else INK)
    ax.set_xticks(range(len(EMOTIONS)), EMOTIONS, rotation=45, ha='right'); ax.set_yticks(range(len(EMOTIONS)), EMOTIONS)
    ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.grid(False)
    ax.set_title(f'{label(FUSION)}: confusion matrix, % of true class\n(mean of {len(mats)} seed(s))')
    fig.colorbar(im, ax=ax, fraction=0.046, label='share of true class')
    plt.show()

## 7 Statistical significance

McNemar's test compares two models on the same test utterances: it only looks at the utterances where exactly
one of them is right. A p-value below 0.05 means the difference is unlikely to be chance.

In [ ]:
if len(mc):
    t = mc.assign(fusion=mc['fusion'].map(label), unimodal=mc['unimodal'].map(label),
                  significant=np.where(mc['p_value'] < 0.05, 'yes', 'no'))
    t = t[['fusion', 'unimodal', 'best_unimodal', 'seed', 'n', 'acc_fusion', 'acc_unimodal',
           'n01 (fusion only right)', 'n10 (unimodal only right)', 'method', 'p_value', 'significant']]
    display(t.style.format({'acc_fusion': '{:.3f}', 'acc_unimodal': '{:.3f}', 'p_value': '{:.3g}'}).hide(axis='index'))
else:
    print('no paired test predictions for the fusion and unimodal runs')

## 8 Robustness

**Transcript condition.** Models trained on Whisper transcripts or on the gold subtitles, each tested on both.
The drop from gold to ASR text at test time is the "ASR tax".
**Noise.** Weighted F1 when white noise is added to the test audio at 20 dB and 10 dB signal-to-noise ratio
(only models that read the waveform).

In [ ]:
tt = ev.transcript_table(test_df)
if len(tt):
    for fam, g in tt.groupby('family'):
        piv = g.pivot(index='train', columns='test', values='weighted_f1').reindex(index=['gold', 'asr'], columns=['gold', 'asr'])
        piv.index = [f'trained on {i}' for i in piv.index]
        piv.columns = [f'tested on {c}' for c in piv.columns]
        display(piv.style.format('{:.3f}', na_rep='-').set_caption(f'{label(fam)}: weighted F1'))
else:
    print('no run evaluated with --text-conditions')

nt = ev.noise_table(test_df)
nt = nt[nt['note'] == ''] if len(nt) else nt
if len(nt):
    conds = ['clean'] + sorted({c.split(' dB')[0] for c in nt.columns if c.endswith('dB wF1')}, key=lambda s: -float(s))
    fig, ax = plt.subplots(figsize=(7, 3.8))
    markers = ['o', 's', 'D', '^', 'v']   # shape as a second cue, so overlapping lines stay distinguishable
    for k, (_, row) in enumerate(nt.iterrows()):
        vals = [row['clean wF1']] + [row[f'{c} dB wF1'] for c in conds[1:]]
        ax.plot(range(len(conds)), vals, '-', marker=markers[k % len(markers)], ms=7, mfc='none', mew=1.6,
                color=SERIES[k % len(SERIES)], label=label(row['model']))
    ax.set_xticks(range(len(conds)), ['clean'] + [f'{c} dB SNR' for c in conds[1:]])
    ax.set_ylabel('test weighted F1'); ax.set_title('Weighted F1 under additive noise')
    ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
    plt.show()
    cols = ['model', 'seeds', 'clean wF1'] + [c for c in nt.columns if c.endswith('dB wF1') or c.endswith('dB delta')]
    display(nt[cols].assign(model=nt['model'].map(label)).style.format(precision=3).hide(axis='index'))
else:
    print('no noise-robustness results (evaluate run --noise-snr 20,10)')

## 9 Ablations and overfitting

**Class-imbalance handling.** Each arm changes only the loss along its whole training chain: unweighted
cross-entropy, inverse-frequency weights, focal loss, or undersampling, against the main recipe
(square-root inverse-frequency weights). An arm is **rejected** when its dev macro F1 falls below the unweighted
baseline, even if its weighted F1 is higher. Scores are on dev; the test split is not used for these choices.

In [ ]:
ARM = {'': 'sqrt-inverse CE (main)', '_cwnone': 'unweighted CE', '_cwinv': 'inverse-frequency CE',
       '_cwfocal': 'focal loss', '_undersample': 'undersampling'}
if len(cw):
    rule = 'M7 rule (dev mF1 >= _cwnone)'
    c = cw.assign(family=cw['name'].map(ev.base_name),
                  arm=cw['name'].map(lambda n: next((v for s, v in ARM.items() if s and n.endswith(s)), ARM[''])))
    fams = list(dict.fromkeys(c['family']))
    arms = [a for a in ARM.values() if a in set(c['arm'])]
    fig, ax = plt.subplots(figsize=(10, 3.8))
    w = 0.8 / len(arms)
    for k, a in enumerate(arms):
        sub = c[c['arm'] == a].set_index('family').reindex(fams)
        xs = np.arange(len(fams)) + (k - (len(arms) - 1) / 2) * w
        bars = ax.bar(xs, sub['dev_macro_f1'], w - 0.02, color=SERIES[k], label=a)
        for b, rej in zip(bars, sub[rule] == 'REJECTED'):
            if rej:
                b.set_hatch('///'); b.set_edgecolor(SURFACE)
    ax.set_xticks(range(len(fams)), [label(f) for f in fams]); ax.grid(axis='x', visible=False)
    ax.set_ylabel('dev macro F1'); ax.set_title('Class-imbalance ablation (hatched = rejected by the macro-F1 rule)')
    ax.legend(ncols=min(3, len(arms)), loc='upper center', bbox_to_anchor=(0.5, -0.12))
    plt.show()
    t = c[['family', 'arm', 'seeds', 'dev_weighted_f1', 'dev_macro_f1', 'test_weighted_f1', 'test_macro_f1', rule]]
    t = t.assign(family=t['family'].map(label)).rename(columns={rule: 'verdict'})
    display(t.style.format(precision=3, na_rep='-').hide(axis='index'))
else:
    print('no ablation runs (stage 6 of scripts/run_all.sh)')

**Dev gain and overfitting.** The dev gain of the fusion model over the best unimodal model (M8), and the gap
between train and dev weighted F1 of each kept checkpoint (M9; a gap above 0.15 flags overfitting).

In [ ]:
if FUSION:
    gg, info = ev.gain_gap_table(test_df, FUSION, UNIMODAL)
    display(gg.assign(model=gg['model'].map(label)).style.format(precision=3).hide(axis='index'))
    if info:
        display(Markdown(f'Dev gain of {label(info["fusion"])} over {label(info["best_unimodal"])}: '
                         f'**{info["dev_gain"]:+.3f}** weighted F1 ({info["M8"]}).'))

## 10 Explanations

Written by `python -m src.explain` for the gated fusion model. **Modality contribution**: how much the predicted
class probability drops when the audio or the text embedding is removed; *audio share* = audio drop / (audio drop
+ text drop), so values above 0.5 mean the prediction leans on audio. **Layer weights**: which WavLM layers the
audio model learned to use. **Case study**: token attributions (Integrated Gradients) and 250 ms audio
occlusion for one test utterance.

In [ ]:
xai = reports / 'xai'
contrib_path = xai / 'modality_contribution.csv'
if contrib_path.exists():
    cdf = pd.read_csv(contrib_path)
    per = cdf.groupby('label').agg(n=('utt_id', 'count'), audio_share=('audio_share', 'mean'),
                                   d_audio=('delta_drop_audio', 'mean'), d_text=('delta_drop_text', 'mean'),
                                   gate=('gate', 'mean')).reindex(EMOTIONS)
    fig, ax = plt.subplots(figsize=(8, 3.4))
    bars = ax.bar(EMOTIONS, per['audio_share'], 0.6, color=MODALITY_COLOR['fusion'])
    for b, v, n in zip(bars, per['audio_share'], per['n']):
        if pd.notna(v):
            ax.text(b.get_x() + b.get_width() / 2, v + 0.02, f'{v:.2f}\n(n={int(n)})', ha='center', fontsize=8.5)
    ax.axhline(0.5, color=INK_2, ls='--', lw=1)
    ax.set_ylim(0, 1.15); ax.set_ylabel('audio share'); ax.grid(axis='x', visible=False)
    ax.set_title('Modality contribution per true class (above 0.5 = audio-driven)')
    plt.show()
    display(per.style.format(precision=3, na_rep='-'))
else:
    print(f'no {contrib_path}; run stage 8 of scripts/run_all.sh')

lw_run = runs_root / 'audio_wavlm_ft' / str(SEED) / 'metrics.json'
if lw_run.exists() and 'layer_weights' in json.loads(lw_run.read_text()):
    lw = np.asarray(json.loads(lw_run.read_text())['layer_weights'])
    fig, ax = plt.subplots(figsize=(8, 2.8))
    ax.bar(range(len(lw)), lw, 0.7, color=MODALITY_COLOR['audio'])
    ax.axhline(1 / len(lw), color=INK_2, ls='--', lw=1)
    ax.set_xticks(range(len(lw))); ax.set_xlabel('WavLM layer (0 = CNN features)'); ax.set_ylabel('softmax weight')
    ax.set_title(f'{label("audio_wavlm_ft")}: learned layer weights (dashed = uniform)'); ax.grid(axis='x', visible=False)
    plt.show()

In [ ]:
cases = sorted((xai / 'cases').glob('*.png')) if (xai / 'cases').exists() else []
if cases:
    display(Markdown(f'Case study `{cases[0].stem}` ({len(cases)} cases in `{xai / "cases"}`)'))
    display(Image(filename=str(cases[0]), width=900))
gt_path = xai / 'global_tokens_ig.csv'
if gt_path.exists():
    gt = pd.read_csv(gt_path)
    top = gt.groupby('class').head(5).groupby('class')['token'].apply(', '.join).reindex(EMOTIONS).dropna()
    display(top.to_frame('top tokens pushing towards the class (aggregated IG)'))

## 11 Quality gates

The project plan (`PLAN.md`) defines pass/fail gates for every phase. These tables are written by the pipeline
itself, with the evidence for each check.

In [ ]:
for name, title in [('gate4_sanity.md', 'Gate 4: training sanity checks'), ('gate5_eval.md', 'Gate 5: evaluation'),
                    ('xai/gate6_xai.md', 'Gate 6: explanations')]:
    p = reports / name
    display(Markdown(f'### {title}'))
    display(Markdown(p.read_text().split('\n', 1)[1] if p.exists() else f'_{p} not written yet_'))

## 12 Reproduce

```bash
uv sync
bash scripts/run_all.sh --device cuda      # full ladder, seeds 13 42 2024 (see README: Full run from a fresh clone)
```

Then re-run this notebook top to bottom (`Run All`). Detail notebooks: `01_eda`, `02_asr_quality`, `03_results`,
`04_xai`.